#Configuration

In [0]:
CATALOG = "cms_medicare_databricks_pipeline"
SOURCE_TABLE = "cms_medicare_databricks_pipeline.silver.readmissions"
TARGET_TABLE = "cms_medicare_databricks_pipeline.gold.hospital_readmission_risk"

#Gold Readmission Risk Table

In [0]:
%sql
CREATE OR REPLACE TABLE cms_medicare_databricks_pipeline.gold.hospital_readmission_risk
CLUSTER BY (condition_name, performance_category)
AS

WITH ca_benchmarks AS (
    SELECT
        measure_name,
        condition_name,
        COUNT(DISTINCT facility_id) AS total_ca_hospitals,
        ROUND(AVG(excess_readmission_ratio), 4) AS ca_avg_excess_ratio,
        ROUND(PERCENTILE(excess_readmission_ratio, 0.25), 4) AS ca_p25_ratio,
        ROUND(PERCENTILE(excess_readmission_ratio, 0.50), 4) AS ca_median_ratio,
        ROUND(PERCENTILE(excess_readmission_ratio, 0.75), 4) AS ca_p75_ratio
    FROM cms_medicare_databricks_pipeline.silver.readmissions
    WHERE excess_readmission_ratio IS NOT NULL
    GROUP BY ALL
),
hospital_ranked AS (
    SELECT
       -- Hospital identity
        r.facility_id,
        r.facility_name,
        r.state,

        -- Condition
        r.measure_name,
        r.condition_name,

        -- Volume
        r.number_of_discharges,
        r.number_of_readmissions,

        -- Core readmission metrics
        r.excess_readmission_ratio,
        -- > 1.0 = more readmissions than CMS expects
        -- < 1.0 = fewer readmissions than CMS expects

        r.predicted_readmission_rate,
        r.expected_readmission_rate,

        -- Measurement period
        r.start_date,
        r.end_date,

        -- Data suppression flag
        -- NULL metrics = CMS withheld data (too few cases)
        CASE
            WHEN r.excess_readmission_ratio IS NULL THEN TRUE
            ELSE FALSE
        END AS is_suppressed,

        -- CA benchmarks for this condition
        b.ca_avg_excess_ratio,
        b.ca_median_ratio,
        b.ca_p25_ratio,
        b.ca_p75_ratio,
        b.total_ca_hospitals,

        -- How far above/below CA average is this hospital?
        ROUND(r.excess_readmission_ratio - b.ca_avg_excess_ratio, 4) AS deviation_from_ca_avg,

        -- Performance category relative to CA peers
        -- NULL = CMS withheld data
        CASE
            WHEN r.excess_readmission_ratio IS NULL
                THEN 'Suppressed'
            WHEN r.excess_readmission_ratio > b.ca_p75_ratio
                THEN 'High Risk'
            -- Above 75th percentile = worse readmission rate than 75% of CA hospitals
            WHEN r.excess_readmission_ratio < b.ca_p25_ratio
                THEN 'Low Risk'
            -- Below 25th percentile = better than 75% of CA hospitals
            ELSE 'Average Risk'
        END AS performance_category,

        -- Rank within California for this condition
        -- 1 = highest readmission ratio = worst performing hospital
        RANK() OVER (PARTITION BY r.measure_name ORDER BY r.excess_readmission_ratio DESC NULLS LAST) AS ca_rank_by_condition,

        -- Audit trail
        r.ingestion_timestamp,
        r.source_file

        FROM cms_medicare_databricks_pipeline.silver.readmissions r
        LEFT JOIN ca_benchmarks b
            ON r.measure_name = b.measure_name
)
SELECT *
FROM hospital_ranked
ORDER BY condition_name, ca_rank_by_condition

#Verify

In [0]:
%sql
SELECT
    condition_name,
    performance_category,
    COUNT(*) AS hospital_count,
    ROUND(AVG(excess_readmission_ratio), 4) AS avg_excess_ratio
FROM cms_medicare_databricks_pipeline.gold.hospital_readmission_risk
GROUP BY ALL
ORDER BY ALL

#Top 10 Highest Risk Hospitals per condition

In [0]:
%sql
--top 10 highest risk hospitals per condition
SELECT
    ca_rank_by_condition AS ca_rank,
    facility_name,
    condition_name,
    excess_readmission_ratio,
    deviation_from_ca_avg,
    performance_category,
    number_of_discharges,
    number_of_readmissions
FROM cms_medicare_databricks_pipeline.gold.hospital_readmission_risk
WHERE performance_category = 'High Risk'
ORDER BY condition_name, ca_rank_by_condition
LIMIT 10

#Summary Stats

In [0]:
%sql
SELECT
    COUNT(DISTINCT facility_id)     AS total_ca_hospitals,
    COUNT(DISTINCT condition_name)  AS conditions_tracked,
    COUNT(*)                        AS total_rows,
    SUM(CASE WHEN performance_category = 'High Risk'
        THEN 1 ELSE 0 END)          AS high_risk_count,
    SUM(CASE WHEN performance_category = 'Low Risk'
        THEN 1 ELSE 0 END)          AS low_risk_count,
    SUM(CASE WHEN performance_category = 'Average Risk'
        THEN 1 ELSE 0 END)          AS average_risk_count,
    SUM(CASE WHEN is_suppressed = TRUE
        THEN 1 ELSE 0 END)          AS suppressed_count
FROM cms_medicare_databricks_pipeline.gold.hospital_readmission_risk